In [24]:
import os
import numpy as np
import random
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, f1_score, roc_curve, auc, confusion_matrix, balanced_accuracy_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import load_model
import seaborn as sns
import gc
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import backend as K
from tensorflow.keras.applications import ResNet50 # ResNet50 was imported but not used, can be kept or removed
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense # These too
import pandas as pd
import shutil # shutil was imported but not used
from collections import defaultdict, deque # deque will be used in largest_tumour_mass

# Set GPU memory growth to avoid allocating all memory at once
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        # Set GPU memory growth
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"Found {len(gpus)} GPU devices and enabled memory growth")
    except RuntimeError as e:
        print(f"GPU setup error: {e}")


SEED = 3888
def set_seed(seed=3888):
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    
set_seed(SEED)

type_to_label_map = {
    'Empty': 0,
    'Non-Tumor': 1,
    'Tumor': 2
}
# Create a reverse map for easy conversion from label index to name
label_idx_to_name_map = {v: k for k, v in type_to_label_map.items()}


UNCENTERED_DATA_DIR = "projectdata/images/uncentred_ternary_224_ALL"
MODEL_PATH = "projectdata/model_h5_files/train_on_centered"

Found 1 GPU devices and enabled memory growth


In [25]:
def preprocess_image(img_path, target_size=(224, 224)):
    img = Image.open(img_path)
    img = img.resize(target_size)
    img_array = np.array(img, dtype=np.float32)
    img_array = img_array / 255.0
    
    return img_array

def load_test_data(dir, quadrant, target_size=(224, 224)):
    """Load test data from a directory without separating by quadrants"""
    X_id = []
    X_test = []
    y_test = []
    

    quadrant_path = os.path.join(dir, quadrant)
    
    # Go through each cell type in the quadrant
    for cell_type in os.listdir(quadrant_path):
        cell_type_path = os.path.join(quadrant_path, cell_type)
        if os.path.isdir(cell_type_path):
            # Check if cell_type is a key in type_to_label_map before proceeding
            if cell_type not in type_to_label_map:
                print(f"Warning: Unknown cell type '{cell_type}' in directory {quadrant_path}. Skipping.")
                continue # Skip this cell_type if it's not in our map
            for filename in os.listdir(cell_type_path):
                if filename.endswith(".png") or filename.endswith(".jpg"):
                    img_path = os.path.join(cell_type_path, filename)
                    img = preprocess_image(img_path, target_size=target_size)
                    
                    parts = filename.replace(".png", "").replace(".jpg", "").split("_")
                    grid_id = f"{parts[1]}_{parts[2]}"

                    X_id.append(grid_id)
                    X_test.append(img)
                    y_test.append(type_to_label_map[cell_type])

    return X_id, np.array(X_test), np.array(y_test)

In [26]:
# This cell (evaluate_model function) will be removed as its logic is integrated into the main loop.
# The original content of this cell is removed.
# K.clear_session() and gc.collect() will be handled in the main loop.
pass 

In [27]:
# This cell's content (the simple loop) will be replaced by a more comprehensive calculation logic.
# The original content of this cell is removed.
pass

In [28]:
# Helper functions and constants

def largest_tumour_mass(grid_IDs, labels):
    # expects grid IDs as [X_coord]_[Y_coord], and label as actual string labels
    tumour_coords = set()
    for id_str, label in zip(grid_IDs, labels):
        if label == "Tumor": # Assuming "Tumor" is the string label for tumor class
            try:
                x, y = map(int, id_str.split("_"))
                tumour_coords.add((x, y))
            except ValueError:
                print(f"Warning: Could not parse grid_ID '{id_str}'. Skipping for largest_tumour_mass.")
                continue


    visited = set()
    # Directions for 4-connectivity. Can be expanded to 8-connectivity if needed.
    directions = [(-1,0), (1,0), (0,-1), (0,1)] 

    def bfs(start_coord):
        queue = deque([start_coord])
        region_coords = set() # Use a set for efficient addition and to store unique coords
        
        if start_coord not in tumour_coords or start_coord in visited:
            return [] # Should not happen if called correctly

        queue.append(start_coord)
        visited.add(start_coord)
        region_coords.add(start_coord)

        while queue:
            cx, cy = queue.popleft()
            for dx, dy in directions:
                neighbor = (cx + dx, cy + dy)
                if neighbor in tumour_coords and neighbor not in visited:
                    visited.add(neighbor)
                    queue.append(neighbor)
                    region_coords.add(neighbor)
        return list(region_coords) # Return as a list of coordinates

    max_len = 0
    for coord in tumour_coords:
        if coord not in visited:
            region = bfs(coord)
            if len(region) > max_len:
                max_len = len(region)
    return max_len

MODEL_NAMES = ["InceptionV3", "VGG19"]
QUADRANTS_STR = ["Q1", "Q2", "Q3", "Q4"]
QUADRANT_MAP_NUM = {"Q1": 1, "Q2": 2, "Q3": 3, "Q4": 4}

# Ensure label_idx_to_name_map is correctly defined if type_to_label_map is primary
if 'label_idx_to_name_map' not in globals():
    label_idx_to_name_map = {v: k for k, v in type_to_label_map.items()}

# Define class names based on the order of labels [0, 1, 2]
# This assumes 0: Empty, 1: Non-Tumor, 2: Tumor from type_to_label_map
CLASS_NAMES = [label_idx_to_name_map[i] for i in range(len(type_to_label_map))]

print("Helper functions and constants defined.")

Helper functions and constants defined.


In [ ]:
all_results_data = []
model_aggregated_cms = {} 
model_quadrant_cms = {} # New: Used to store CM for each quadrant of each model

for model_name in MODEL_NAMES:
    print(f"\nProcessing Model: {model_name}")
    
    model_accuracies = []
    model_f1_scores = []
    model_true_tumor_counts = []
    model_pred_tumor_counts = []
    model_true_nontumor_counts = []
    model_pred_nontumor_counts = []
    model_true_empty_counts = []
    model_pred_empty_counts = []
    model_percent_tumor_true_list = []
    model_percent_tumor_pred_list = []
    model_largest_mass_true_list = []
    model_largest_mass_pred_list = []
    model_rmse_percent_tumor_list = []
    model_rmse_largest_mass_list = []
    model_rmse_tumor_count_list = []

    aggregated_cm_model = np.zeros((len(CLASS_NAMES), len(CLASS_NAMES)), dtype=int)
    model_quadrant_cms[model_name] = {} # Initialize quadrant CM dictionary for the current model

    for quadrant_str in QUADRANTS_STR:
        print(f"  Processing Quadrant: {quadrant_str}")
        quadrant_num = QUADRANT_MAP_NUM[quadrant_str]

        target_size = (299, 299) if model_name == "InceptionV3" else (224, 224)
        
        X_ids, X_test, y_test = load_test_data(UNCENTERED_DATA_DIR, quadrant_str, target_size=target_size)

        if X_test.size == 0:
            print(f"    No data loaded for {model_name} - {quadrant_str}. Skipping.")
            continue
            
        if model_name == "InceptionV3":
            model_filename = f"inceptionV3_Tanvi_fold_{quadrant_num}.h5"
        elif model_name == "VGG19":
            model_filename = f"VGG19_fold_{quadrant_num}.h5"
        else:
            raise ValueError(f"Unknown model name: {model_name}")
            
        model_file_path = os.path.join(MODEL_PATH, model_filename)
        
        if not os.path.exists(model_file_path):
            print(f"    Model file not found: {model_file_path}. Skipping quadrant.")
            continue

        K.clear_session()
        gc.collect()
        model = load_model(model_file_path)
        
        y_prob = model.predict(X_test)
        y_pred = np.argmax(y_prob, axis=1)

        # Metrics Calculation
        accuracy = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
        
        y_test_str = [label_idx_to_name_map[i] for i in y_test]
        y_pred_str = [label_idx_to_name_map[i] for i in y_pred]

        true_tumor_count = y_test_str.count("Tumor")
        pred_tumor_count = y_pred_str.count("Tumor")
        true_nontumor_count = y_test_str.count("Non-Tumor")
        pred_nontumor_count = y_pred_str.count("Non-Tumor")
        true_empty_count = y_test_str.count("Empty")
        pred_empty_count = y_pred_str.count("Empty")

        total_grids_true = len(y_test_str)
        total_grids_pred = len(y_pred_str) 

        percent_tumor_true = (true_tumor_count / total_grids_true) * 100 if total_grids_true > 0 else 0
        percent_tumor_pred = (pred_tumor_count / total_grids_pred) * 100 if total_grids_pred > 0 else 0
        
        largest_mass_true = largest_tumour_mass(X_ids, y_test_str)
        largest_mass_pred = largest_tumour_mass(X_ids, y_pred_str)

        rmse_percent_tumor = np.sqrt((percent_tumor_true - percent_tumor_pred)**2)
        rmse_largest_mass = np.sqrt((largest_mass_true - largest_mass_pred)**2)
        rmse_tumor_count = np.sqrt((true_tumor_count - pred_tumor_count)**2)
        
        cm = confusion_matrix(y_test, y_pred, labels=list(range(len(CLASS_NAMES))))
        if cm.shape != (len(CLASS_NAMES), len(CLASS_NAMES)):
            print(f"Warning: CM shape is {cm.shape}, expected {(len(CLASS_NAMES), len(CLASS_NAMES))}. Check data for {quadrant_str}.")
        
        aggregated_cm_model += cm
        model_quadrant_cms[model_name][quadrant_str] = cm 

        # Store results for this quadrant
        quadrant_results = {
            "Model": model_name,
            "Quadrant": quadrant_str,
            "Accuracy": accuracy,
            "F1_Score": f1,
            "True_Tumor_Count": true_tumor_count,
            "Pred_Tumor_Count": pred_tumor_count,
            "True_NonTumor_Count": true_nontumor_count,
            "Pred_NonTumor_Count": pred_nontumor_count,
            "True_Empty_Count": true_empty_count,
            "Pred_Empty_Count": pred_empty_count,
            "Percent_Tumor_True": percent_tumor_true,
            "Percent_Tumor_Pred": percent_tumor_pred,
            "Largest_Mass_True": largest_mass_true,
            "Largest_Mass_Pred": largest_mass_pred,
            "RMSE_Percent_Tumor": rmse_percent_tumor,
            "RMSE_Largest_Mass": rmse_largest_mass,
            "RMSE_Tumor_Count": rmse_tumor_count,
        }
        
        for r_idx in range(cm.shape[0]):
            for c_idx in range(cm.shape[1]):
                quadrant_results[f"CM_True{CLASS_NAMES[r_idx]}_Pred{CLASS_NAMES[c_idx]}"] = cm[r_idx, c_idx]
        all_results_data.append(quadrant_results)

        # Append to model lists for averaging
        model_accuracies.append(accuracy)
        model_f1_scores.append(f1)
        model_true_tumor_counts.append(true_tumor_count)
        model_pred_tumor_counts.append(pred_tumor_count)
        model_true_nontumor_counts.append(true_nontumor_count)
        model_pred_nontumor_counts.append(pred_nontumor_count)
        model_true_empty_counts.append(true_empty_count)
        model_pred_empty_counts.append(pred_empty_count)
        model_percent_tumor_true_list.append(percent_tumor_true)
        model_percent_tumor_pred_list.append(percent_tumor_pred)
        model_largest_mass_true_list.append(largest_mass_true)
        model_largest_mass_pred_list.append(largest_mass_pred)
        model_rmse_percent_tumor_list.append(rmse_percent_tumor)
        model_rmse_largest_mass_list.append(rmse_largest_mass)
        model_rmse_tumor_count_list.append(rmse_tumor_count)
        
        print(f"    Confusion Matrix for {model_name} - {quadrant_str}:\n{cm}")

        del model, X_test, y_test, y_pred, y_prob, X_ids
        K.clear_session()
        gc.collect()

    # Calculate and store average metrics for the current model
    if model_accuracies: 
        avg_results = {
            "Model": model_name,
            "Quadrant": "Average",
            "Accuracy": np.mean(model_accuracies),
            "F1_Score": np.mean(model_f1_scores),
            "True_Tumor_Count": np.mean(model_true_tumor_counts),
            "Pred_Tumor_Count": np.mean(model_pred_tumor_counts),
            "True_NonTumor_Count": np.mean(model_true_nontumor_counts),
            "Pred_NonTumor_Count": np.mean(model_pred_nontumor_counts),
            "True_Empty_Count": np.mean(model_true_empty_counts),
            "Pred_Empty_Count": np.mean(model_pred_empty_counts),
            "Percent_Tumor_True": np.mean(model_percent_tumor_true_list),
            "Percent_Tumor_Pred": np.mean(model_percent_tumor_pred_list),
            "Largest_Mass_True": np.mean(model_largest_mass_true_list),
            "Largest_Mass_Pred": np.mean(model_largest_mass_pred_list),
            "RMSE_Percent_Tumor": np.mean(model_rmse_percent_tumor_list),
            "RMSE_Largest_Mass": np.mean(model_rmse_largest_mass_list),
            "RMSE_Tumor_Count": np.mean(model_rmse_tumor_count_list),
        }
    
        avg_results["Accuracy_Std"] = np.std(model_accuracies)
        avg_results["F1_Score_Std"] = np.std(model_f1_scores)
        
        all_results_data.append(avg_results)
        
        model_aggregated_cms[model_name] = aggregated_cm_model # Store aggregated CM

        print(f"  Aggregated Confusion Matrix for {model_name} (Sum over quadrants):\n{aggregated_cm_model}")
        print(f"  Average Metrics for {model_name}:")
        for key, value in avg_results.items():
            if key not in ["Model", "Quadrant"]: # and not key.startswith("CM_Aggregated"):
                 print(f"    {key}: {value:.4f}" if isinstance(value, float) else f"    {key}: {value}")
    else:
        print(f"  No data processed for model {model_name}, cannot calculate averages.")

print("\nAll processing finished.")


Processing Model: InceptionV3
  Processing Quadrant: Q1
32/32 [==============================] - 4s 84ms/step
    Confusion Matrix for InceptionV3 - Q1:
[[  5  70   0]
 [  3 613   2]
 [  0 128 183]]
    Confusion Matrix for InceptionV3 - Q1:
[[  5  70   0]
 [  3 613   2]
 [  0 128 183]]
  Processing Quadrant: Q2
  Processing Quadrant: Q2
37/37 [==============================] - 4s 82ms/step
    Confusion Matrix for InceptionV3 - Q2:
[[ 14 151   5]
 [ 19 682  35]
 [  0  73 195]]
    Confusion Matrix for InceptionV3 - Q2:
[[ 14 151   5]
 [ 19 682  35]
 [  0  73 195]]
  Processing Quadrant: Q3
  Processing Quadrant: Q3
55/55 [==============================] - 6s 82ms/step
    Confusion Matrix for InceptionV3 - Q3:
[[ 22   8   0]
 [ 12 413  13]
 [  2 323 946]]
    Confusion Matrix for InceptionV3 - Q3:
[[ 22   8   0]
 [ 12 413  13]
 [  2 323 946]]
  Processing Quadrant: Q4
  Processing Quadrant: Q4
63/63 [==============================] - 7s 82ms/step
    Confusion Matrix for InceptionV3 

In [ ]:
# Convert results to DataFrame and save to CSV
if all_results_data:
    results_df = pd.DataFrame(all_results_data)
    
    # Define column order for better readability
    base_cols = ["Model", "Quadrant", "Accuracy", "F1_Score", 
                 "True_Tumor_Count", "Pred_Tumor_Count", 
                 "True_NonTumor_Count", "Pred_NonTumor_Count",
                 "True_Empty_Count", "Pred_Empty_Count",
                 "Percent_Tumor_True", "Percent_Tumor_Pred",
                 "Largest_Mass_True", "Largest_Mass_Pred",
                 "RMSE_Percent_Tumor", "RMSE_Largest_Mass", "RMSE_Tumor_Count"]
    
    std_cols = ["Accuracy_Std", "F1_Score_Std"] # Keep only standard deviations for Accuracy and F1

    all_generated_cols = list(results_df.columns)
    
    ordered_columns = []
    present_cols_set = set(all_generated_cols)

    # Column sorting no longer includes CM columns for each quadrant
    for col_group in [base_cols, std_cols]: # Removed cm_cols_quadrant
        for col in col_group:
            if col in present_cols_set and col not in ordered_columns:
                ordered_columns.append(col)
    
    for col in all_generated_cols:
        if col not in ordered_columns:
            ordered_columns.append(col)
            
    results_df = results_df[ordered_columns]
    
    csv_filename = "aggregated_model_evaluation_results.csv"
    results_df.to_csv(csv_filename, index=False, float_format='%.4f')
    print(f"\nResults saved to {csv_filename}")
    display(results_df)

    # Additionally print aggregated confusion matrices below the CSV
    if model_aggregated_cms:
        print("\n\n--- Aggregated Confusion Matrices (Sum over Quadrants) ---")
        for model_name_key, agg_cm in model_aggregated_cms.items():
            print(f"\nModel: {model_name_key}")
            cm_df = pd.DataFrame(agg_cm, index=CLASS_NAMES, columns=CLASS_NAMES)
            print(cm_df)

    # New: Print confusion matrix for each quadrant of each model
    if model_quadrant_cms:
        print("\n\n--- Individual Quadrant Confusion Matrices ---")
        for model_name_key, quadrant_cms_dict in model_quadrant_cms.items():
            print(f"\nModel: {model_name_key}")
            for quadrant_key, q_cm in quadrant_cms_dict.items():
                print(f"  Quadrant: {quadrant_key}")
                cm_df = pd.DataFrame(q_cm, index=CLASS_NAMES, columns=CLASS_NAMES)
                print(cm_df)
                if not (model_name_key == list(model_quadrant_cms.keys())[-1] and quadrant_key == list(quadrant_cms_dict.keys())[-1]):
                    print("-" * 30) # Add separator unless it's the very last CM
            
else:
    print("No results were generated to save to CSV.")


Results saved to aggregated_model_evaluation_results.csv


,Model,Quadrant,Accuracy,F1_Score,True_Tumor_Count,Pred_Tumor_Count,True_NonTumor_Count,Pred_NonTumor_Count,True_Empty_Count,Pred_Empty_Count,Percent_Tumor_True,Percent_Tumor_Pred,Largest_Mass_True,Largest_Mass_Pred,RMSE_Percent_Tumor,RMSE_Largest_Mass,RMSE_Tumor_Count,Accuracy_Std,F1_Score_Std
0,InceptionV3,Q1,0.797809,0.765670,311.00,185.0,618.0,811.00,75.0,8.00,30.976096,18.426295,249.00,58.00,12.549801,191.00,126.00,NaN,NaN
1,InceptionV3,Q2,0.758944,0.717745,268.00,235.0,736.0,906.00,170.0,33.00,22.827939,20.017036,145.00,84.00,2.810903,61.00,33.00,NaN,NaN
2,InceptionV3,Q3,0.794135,0.807611,1271.00,959.0,438.0,744.00,30.0,36.00,73.087982,55.146636,1216.00,678.00,17.941346,538.00,312.00,NaN,NaN
3,InceptionV3,Q4,0.749500,0.722540,565.00,365.0,1208.0,1549.00,227.0,86.00,28.250000,18.250000,109.00,89.00,10.000000,20.00,200.00,NaN,NaN
4,InceptionV3,Average,0.775097,0.753391,603.75,436.0,750.0,1002.50,125.5,40.75,38.785504,27.959992,429.75,227.25,10.825512,202.50,167.75,0.021180,0.036445
5,VGG19,Q1,0.777888,0.754987,311.00,223.0,618.0,765.00,75.0,16.00,30.976096,22.211155,249.00,60.00,8.764940,189.00,88.00,NaN,NaN
6,VGG19,Q2,0.729983,0.706305,268.00,199.0,736.0,894.00,170.0,81.00,22.827939,16.950596,145.00,63.00,5.877342,82.00,69.00,NaN,NaN
7,VGG19,Q3,0.814836,0.826909,1271.00,994.0,438.0,702.00,30.0,43.00,73.087982,57.159287,1216.00,848.00,15.928695,368.00,277.00,NaN,NaN
8,VGG19,Q4,0.738000,0.735529,565.00,398.0,1208.0,1370.00,227.0,232.00,28.250000,19.900000,109.00,67.00,8.350000,42.00,167.00,NaN,NaN
9,VGG19,Average,0.765177,0.755932,603.75,453.5,750.0,932.75,125.5,93.00,38.785504,29.055260,429.75,259.50,9.730244,170.25,150.25,0.033929,0.044491




--- Aggregated Confusion Matrices (Sum over Quadrants) ---

Model: InceptionV3
           Empty  Non-Tumor  Tumor
Empty         70        426      6
Non-Tumor     86       2839     75
Tumor          7        745   1663

Model: VGG19
           Empty  Non-Tumor  Tumor
Empty        163        332      7
Non-Tumor    199       2681    120
Tumor         10        718   1687


--- Individual Quadrant Confusion Matrices ---

Model: InceptionV3
  Quadrant: Q1
           Empty  Non-Tumor  Tumor
Empty          5         70      0
Non-Tumor      3        613      2
Tumor          0        128    183
------------------------------
  Quadrant: Q2
           Empty  Non-Tumor  Tumor
Empty         14        151      5
Non-Tumor     19        682     35
Tumor          0         73    195
------------------------------
  Quadrant: Q3
           Empty  Non-Tumor  Tumor
Empty         22          8      0
Non-Tumor     12        413     13
Tumor          2        323    946
-----------------------------